# Week 9 demos

In [2]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import linregress

## Loading and inspect netcdf data

In [3]:
# loading NOAA ice concentration data
data = xr.open_dataset('data/icec.mnmean.nc')
display(data)

<xarray.Dataset> Size: 128MB
Dimensions:    (time: 494, lat: 180, lon: 360, nbnds: 2)
Coordinates:
  * lat        (lat) float32 720B 89.5 88.5 87.5 86.5 ... -87.5 -88.5 -89.5
  * lon        (lon) float32 1kB 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * time       (time) datetime64[ns] 4kB 1981-12-01 1982-01-01 ... 2023-01-01
Dimensions without coordinates: nbnds
Data variables:
    icec       (time, lat, lon) float32 128MB ...
    time_bnds  (time, nbnds) datetime64[ns] 8kB ...
Attributes:
    title:          NOAA Optimum Interpolation (OI) SST V2
    Conventions:    CF-1.0
    history:        Wed Apr  6 13:51:42 2005: ncks -d time,0,278 SAVEs/icec.m...
    comments:       Data described in  Reynolds, R.W., N.A. Rayner, T.M.\nSmi...
    platform:       Model
    source:         NCEP Climate Modeling Branch
    institution:    National Centers for Environmental Prediction
    References:     https://www.psl.noaa.gov/data/gridded/data.noaa.oisst.v2....
    NCO:            4.0.0
    dataset_title:  NOAA Optimum Interpolation (OI) SST V2
    source_url:     http://www.emc.ncep.noaa.gov/research/cmb/sst_analysis/

In [ ]:
# extract monthly means of ice concentration DataArray from the Dataset
ice = data.icec
display(ice)

## Dimensional reduction on xarray DataArray

### Poll #1 (think-pair-share; Jupyter **Off**)

Which of the following codes can be used to reduce the dimension of the sea ice concentration data by making an average over sea ice concentration in the part of the globe where latitude is above 45°N?

1. ```python
   ice.mean('lat').mean('lon')
   ```
   
2. ```python
   ice.sel(lat = ice.lat > 45).mean('lat')
   ```

3. ```python
   ice.sel(lat = ice.lat > 45).mean('lon')
   ```

4. ```python
   ice.sel(lat = ice.lat > 45).mean(['lat', 'lon'])
   ```

## Conversion to pandas dataframe

In [ ]:
# convert the time series from xarray to pandas
ice_df = ice_mean.to_dataframe().reset_index()
display(ice_df)

In [ ]:
ice_df.info()

## Plotting the time series

In [ ]:
# Starter code

fig = plt.figure()
ax = fig.add_subplot()

plt.show(fig)

### Think-pair-share

What have you noticed from the plot above?

## Linear regression

In [ ]:
# convert the timestamp to floating point numbers
ice_time_day = (ice_df.time - ice_df.time[0]) / np.timedelta64(1, 'D')

### Your turn #1

Perform the linear regression of mean ice concentration against time, then print and inspect the results (intercept, slope, R<sup>2</sup> and r)

In [ ]:
# Your code here
ice_trend = ...

### Think-pair-share

How would you interpret the results from the linear fit?

### Your turn #2

Add the regression line to the previous plot. Make sure you follow best practices for good plot

In [ ]:
# Create the y (ice concentration) coordindates of the regression line
ice_fit = ice_trend.intercept + ice_trend.slope * ice_time_day

In [ ]:
# Your code here


## Grouped summarization: climatology and anomaly

In [ ]:
# Climatology
# Take the monthly mean (e.g. Mean ice concentration in January for all years, february for all years, etc.)
ice_mean_climatology = ice_df["icec"].groupby(ice_df.time.dt.month).mean()
display(ice_mean_climatology)

In [ ]:
# Anomalies
# These are deviations from the mean seasonal cycle
ice_mean_anomalies = ice_df["icec"] - ice_mean_climatology.loc[ice_df.time.dt.month].values
display(ice_mean_anomalies)

In [ ]:
fig = plt.figure(figsize = (10, 4))
ax = fig.add_subplot()

ax.plot(np.arange(1, 13, 1), ice_mean_climatology, lw=2)
ax.set_xlabel('Month')
ax.set_ylabel('Sea Ice Concentration (%)')
ax.set_title('Arctic Averaged Sea Ice Concentration Mean Seasonal Cycle')
ax.grid()

plt.show(fig)

## Multi-panel plot

In [ ]:
fig = plt.figure(figsize=(12, 9))

# Upper panel: sea ice concentration and trend

# Lower panel: sea ice concentration anomalies

plt.show(fig)

## _(Optional) redo linear regression on anomaly data_

### Think Pair Share

Compare the result of _this_ linear regression to the previous one, what have you noticed? Does the result make sense?

## _(Optional) a more complicated multi-panel plot_

In [ ]:
# trick: insert nan between month = 12 and month = 1

tmp = np.array([ice_df.time.dt.month.values, ice_df.icec.values]).T
ice_month = np.array([[0, 0]])

for row in tmp:
    ice_month = np.append(ice_month, [row], axis=0)
    if row[0] == 12:
        ice_month = np.append(ice_month, [[np.nan, np.nan]], axis=0)

ice_month = ice_month[1:]

In [ ]:
fig = plt.figure(figsize=(12, 6))

# Top panel: sea ice concentration

ax0 = fig.add_subplot(3, 2, (1, 2))

ax0.plot(ice_df.time, ice_df.icec)
ax0.set_title('Arctic Averaged Monthly Sea Ice Concentrations')
ax0.set_ylabel('Sea Ice Concentration (%)')
ax0.grid()

# Lower left: climatology

ax1 = fig.add_subplot(3, 2, (3, 5))

ax1.plot(ice_month[:, 0], ice_month[:, 1], lw=0.5, color="darkgray" )
ax1.plot(np.arange(1, 13, 1), ice_mean_climatology, lw=2)
ax1.set_xlabel('Month')
ax1.set_ylabel('Sea Ice Concentration (%)')
ax1.set_title('Arctic Averaged Sea Ice Concentration Mean Seasonal Cycle')
ax1.grid()

# Lower right: sea ice concentration anomalies

ax2 = fig.add_subplot(3, 2, (4, 6))

ax2.plot(ice_df.time, ice_mean_anomalies)
ax2.plot(ice_df.time, ice_anom_fit, label='Trend Line', color='red', ls = '--', lw = 2)
ax2.set_title('Arctic Averaged Monthly Sea Ice Concentration Anomalies')
ax2.set_xlabel('Date')
ax2.set_ylabel('Sea Ice Concentration Anomaly (%)')
ax2.axhline(y = 0, color = 'k', ls = '--') # Helpful for anomalies to have reference line
ax2.set_ylim(-7,5)
ax2.legend()
ax2.grid()

fig.tight_layout(pad=3.0, w_pad=0.5, h_pad=1.0)
fig.savefig("ice_illustration.png")
plt.show(fig)